# AHR AIMO Graph Prototype

This notebook turns the reverse-solver AST generator into a first graph-native AHR experiment for AIMO-style integer reasoning.

The prototype has four goals:

1. Generate synthetic integer puzzles as typed computation graphs.
2. Tensorize those graphs into a variable-size batched representation.
3. Train a compact AHR-like model to recover the masked variable assignments.
4. Inspect where the model succeeds and where it still behaves like a shallow propagator.

This is intentionally not yet a full competition solver. The missing pieces for a serious AIMO system are still:
- LaTeX parsing into graph candidates
- symbolic tool use during inference
- candidate branching and reranking
- problem-family routing for geometry / number theory / combinatorics

What this notebook *does* provide is a clean graph-reasoning core that can become the middle of that larger system.

In [ ]:
import importlib
import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional


def require_package(import_name: str, pip_name: Optional[str] = None):
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError as exc:
        package = pip_name or import_name
        raise RuntimeError(
            f"Missing dependency '{package}'. Install it into the active kernel before running this notebook. "
            "For local runs, install at least: numpy torch networkx sympy z3-solver. "
            "On Kaggle, PyTorch and NumPy are usually already present."
        ) from exc


np = require_package("numpy")
torch = require_package("torch")
networkx = require_package("networkx")
sympy = require_package("sympy")
z3 = require_package("z3", "z3-solver")

nn = torch.nn
F = require_package("torch.nn.functional")
from torch.utils.data import DataLoader, Dataset

from generate_reverse_ast_puzzles import GeneratorConfig, ReverseASTPuzzleGenerator

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("torch:", torch.__version__)


## Synthetic Graph Curriculum

We start with synthetic reverse-generated puzzles because they give us exact solutions, guaranteed integer structure, and a verifier-backed curriculum. The model sees only the masked graph, while the supervision is the hidden variable assignment.

In [ ]:
@dataclass
class DataConfig:
    train_puzzles: int = 256
    valid_puzzles: int = 64
    min_vars: int = 3
    max_vars: int = 5
    value_min: int = 1
    value_max: int = 30
    coeff_max: int = 5
    extra_equations_max: int = 2
    nonlinear_probability: float = 0.60
    solver_timeout_ms: int = 1500


DATA_CFG = DataConfig()
MAX_VARS = DATA_CFG.max_vars
MAX_VALUE = DATA_CFG.value_max


def build_split(num_puzzles: int, seed: int):
    cfg = GeneratorConfig(
        num_puzzles=num_puzzles,
        min_vars=DATA_CFG.min_vars,
        max_vars=DATA_CFG.max_vars,
        value_min=DATA_CFG.value_min,
        value_max=DATA_CFG.value_max,
        coeff_min=1,
        coeff_max=DATA_CFG.coeff_max,
        offset_abs_max=12,
        extra_equations_max=DATA_CFG.extra_equations_max,
        nonlinear_probability=DATA_CFG.nonlinear_probability,
        max_attempts_per_puzzle=200,
        solver_timeout_ms=DATA_CFG.solver_timeout_ms,
        seed=seed,
        output_path=Path("outputs") / f"tmp_graph_split_{seed}.json",
    )
    generator = ReverseASTPuzzleGenerator(cfg)
    return generator.generate_batch()


train_raw = build_split(DATA_CFG.train_puzzles, SEED)
valid_raw = build_split(DATA_CFG.valid_puzzles, SEED + 1)

print(f"train puzzles: {len(train_raw)}")
print(f"valid puzzles: {len(valid_raw)}")
print(json.dumps({
    "puzzle_id": train_raw[0]["puzzle_id"],
    "solution": train_raw[0]["solution"],
    "metadata": train_raw[0]["metadata"],
    "equations": train_raw[0]["equations"][:2],
}, indent=2))


## Graph Tensorization

The notebook uses a minimal typed-graph encoding:
- node ids become padded batched positions
- node kinds / operators / equation types become categorical ids
- scalar metadata becomes numeric side features
- adjacency is kept as dense child-to-parent matrices for simplicity

For a larger run, you would likely switch this to sparse tensors or a packed edge list.

In [ ]:
NODE_KIND_TO_ID = {
    "<pad>": 0,
    "variable": 1,
    "constant": 2,
    "operator": 3,
    "equation": 4,
}

OP_TO_ID = {
    None: 0,
    "add": 1,
    "sub": 2,
    "mul": 3,
    "eq": 4,
}

EQ_TYPE_TO_ID = {
    None: 0,
    "linear_backbone": 1,
    "linear_decorative": 2,
    "nonlinear_decorative": 3,
}

NUMERIC_FEATURE_DIM = 8


def variable_slot(name: Optional[str]) -> int:
    if not name or not name.startswith("x"):
        return -1
    return int(name[1:])


def tensorize_puzzle(puzzle: Dict) -> Dict[str, np.ndarray]:
    nodes = puzzle["graph"]["nodes"]
    edges = puzzle["graph"]["edges"]
    node_index = {node["id"]: idx for idx, node in enumerate(nodes)}
    num_nodes = len(nodes)

    kind_ids = np.zeros(num_nodes, dtype=np.int64)
    op_ids = np.zeros(num_nodes, dtype=np.int64)
    eq_type_ids = np.zeros(num_nodes, dtype=np.int64)
    numeric = np.zeros((num_nodes, NUMERIC_FEATURE_DIM), dtype=np.float32)

    variable_positions = np.full(MAX_VARS, -1, dtype=np.int64)
    targets = np.zeros(MAX_VARS, dtype=np.int64)
    target_mask = np.zeros(MAX_VARS, dtype=np.float32)

    for idx, node in enumerate(nodes):
        kind_ids[idx] = NODE_KIND_TO_ID[node["kind"]]
        op_ids[idx] = OP_TO_ID[node["op"]]
        eq_type_ids[idx] = EQ_TYPE_TO_ID[node["equation_type"]]

        raw_value = 0 if node["value"] is None else int(node["value"])
        slot = variable_slot(node["name"])

        numeric[idx] = np.array([
            float(node["masked"]),
            float(node["revealed"]),
            float(node["value"] is not None),
            float(raw_value) / max(MAX_VALUE, 1),
            float(node["constant_role"] == "rhs"),
            float(node["constant_role"] == "coefficient"),
            0.0 if slot < 0 else float(slot + 1) / MAX_VARS,
            1.0,
        ], dtype=np.float32)

        if node["kind"] == "variable":
            assert slot >= 0
            variable_positions[slot] = idx
            targets[slot] = int(puzzle["solution"][node["name"]])
            target_mask[slot] = 1.0

    adj = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    rev_adj = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    left_adj = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    right_adj = np.zeros((num_nodes, num_nodes), dtype=np.float32)

    for edge in edges:
        source = node_index[edge["source"]]
        target = node_index[edge["target"]]
        adj[source, target] = 1.0
        rev_adj[target, source] = 1.0
        if edge["slot"] == "left":
            left_adj[source, target] = 1.0
        elif edge["slot"] == "right":
            right_adj[source, target] = 1.0

    return {
        "kind_ids": kind_ids,
        "op_ids": op_ids,
        "eq_type_ids": eq_type_ids,
        "numeric": numeric,
        "adj": adj,
        "rev_adj": rev_adj,
        "left_adj": left_adj,
        "right_adj": right_adj,
        "variable_positions": variable_positions,
        "targets": targets,
        "target_mask": target_mask,
        "num_nodes": np.int64(num_nodes),
    }


class PuzzleDataset(Dataset):
    def __init__(self, raw_puzzles: List[Dict]):
        self.raw_puzzles = raw_puzzles
        self.items = [tensorize_puzzle(puzzle) for puzzle in raw_puzzles]

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> Dict[str, np.ndarray]:
        return self.items[idx]


train_ds = PuzzleDataset(train_raw)
valid_ds = PuzzleDataset(valid_raw)


def collate_graph_batch(samples: List[Dict[str, np.ndarray]]) -> Dict[str, torch.Tensor]:
    batch_size = len(samples)
    max_nodes = max(int(sample["num_nodes"]) for sample in samples)

    kind_ids = torch.zeros(batch_size, max_nodes, dtype=torch.long)
    op_ids = torch.zeros(batch_size, max_nodes, dtype=torch.long)
    eq_type_ids = torch.zeros(batch_size, max_nodes, dtype=torch.long)
    numeric = torch.zeros(batch_size, max_nodes, NUMERIC_FEATURE_DIM, dtype=torch.float32)
    node_mask = torch.zeros(batch_size, max_nodes, dtype=torch.float32)

    adj = torch.zeros(batch_size, max_nodes, max_nodes, dtype=torch.float32)
    rev_adj = torch.zeros(batch_size, max_nodes, max_nodes, dtype=torch.float32)
    left_adj = torch.zeros(batch_size, max_nodes, max_nodes, dtype=torch.float32)
    right_adj = torch.zeros(batch_size, max_nodes, max_nodes, dtype=torch.float32)

    variable_positions = torch.full((batch_size, MAX_VARS), -1, dtype=torch.long)
    targets = torch.zeros(batch_size, MAX_VARS, dtype=torch.long)
    target_mask = torch.zeros(batch_size, MAX_VARS, dtype=torch.float32)

    for batch_idx, sample in enumerate(samples):
        num_nodes = int(sample["num_nodes"])

        kind_ids[batch_idx, :num_nodes] = torch.from_numpy(sample["kind_ids"])
        op_ids[batch_idx, :num_nodes] = torch.from_numpy(sample["op_ids"])
        eq_type_ids[batch_idx, :num_nodes] = torch.from_numpy(sample["eq_type_ids"])
        numeric[batch_idx, :num_nodes] = torch.from_numpy(sample["numeric"])
        node_mask[batch_idx, :num_nodes] = 1.0

        adj[batch_idx, :num_nodes, :num_nodes] = torch.from_numpy(sample["adj"])
        rev_adj[batch_idx, :num_nodes, :num_nodes] = torch.from_numpy(sample["rev_adj"])
        left_adj[batch_idx, :num_nodes, :num_nodes] = torch.from_numpy(sample["left_adj"])
        right_adj[batch_idx, :num_nodes, :num_nodes] = torch.from_numpy(sample["right_adj"])

        variable_positions[batch_idx] = torch.from_numpy(sample["variable_positions"])
        targets[batch_idx] = torch.from_numpy(sample["targets"])
        target_mask[batch_idx] = torch.from_numpy(sample["target_mask"])

    return {
        "kind_ids": kind_ids,
        "op_ids": op_ids,
        "eq_type_ids": eq_type_ids,
        "numeric": numeric,
        "node_mask": node_mask,
        "adj": adj,
        "rev_adj": rev_adj,
        "left_adj": left_adj,
        "right_adj": right_adj,
        "variable_positions": variable_positions,
        "targets": targets,
        "target_mask": target_mask,
    }


train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_graph_batch)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False, collate_fn=collate_graph_batch)

example_batch = next(iter(train_loader))
for key, value in example_batch.items():
    print(key, tuple(value.shape), value.dtype)


## AHR-Math Prototype

This model keeps the same broad idea as the Sudoku AHR:
- a fast worker repeatedly updates local node states
- a slower queen summarizes global proof state
- variable predictions are read from the hidden states of the variable nodes

The main change is that everything now lives on a typed computation graph instead of a 9x9 board.

In [ ]:
class GraphWorkerLayer(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5 * d_model, 2 * d_model),
            nn.GELU(),
            nn.Linear(2 * d_model, d_model),
        )
        self.norm = nn.LayerNorm(d_model)

    def aggregate(self, adjacency: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        # Adjacency is stored as child -> parent. The einsum sums over source nodes,
        # producing one aggregated context vector per target node.
        ctx = torch.einsum("bij,bid->bjd", adjacency, h)
        deg = adjacency.sum(dim=1).clamp_min(1.0).unsqueeze(-1)
        return ctx / deg

    def forward(
        self,
        h: torch.Tensor,
        adj: torch.Tensor,
        rev_adj: torch.Tensor,
        left_adj: torch.Tensor,
        right_adj: torch.Tensor,
        node_mask: torch.Tensor,
    ) -> torch.Tensor:
        child_ctx = self.aggregate(adj, h)
        parent_ctx = self.aggregate(rev_adj, h)
        left_ctx = self.aggregate(left_adj, h)
        right_ctx = self.aggregate(right_adj, h)

        out = self.net(torch.cat([h, child_ctx, parent_ctx, left_ctx, right_ctx], dim=-1))
        out = self.norm(h + out)
        return out * node_mask.unsqueeze(-1)


class QueenSummaryLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 2 * d_model),
            nn.GELU(),
            nn.Linear(2 * d_model, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        attn_out, _ = self.attn(tokens, tokens, tokens, need_weights=False)
        x = self.norm1(tokens + attn_out)
        x = self.norm2(x + self.ff(x))
        return x


class AHRMathPrototype(nn.Module):
    def __init__(
        self,
        d_model: int = 128,
        num_worker_layers: int = 3,
        num_heads: int = 4,
        queen_period: int = 2,
        value_max: int = MAX_VALUE,
    ):
        super().__init__()
        self.d_model = d_model
        self.queen_period = queen_period
        self.value_max = value_max

        self.kind_emb = nn.Embedding(len(NODE_KIND_TO_ID), d_model)
        self.op_emb = nn.Embedding(len(OP_TO_ID), d_model)
        self.eq_type_emb = nn.Embedding(len(EQ_TYPE_TO_ID), d_model)
        self.numeric_proj = nn.Sequential(
            nn.Linear(NUMERIC_FEATURE_DIM, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.workers = nn.ModuleList([GraphWorkerLayer(d_model) for _ in range(num_worker_layers)])
        self.queen = QueenSummaryLayer(d_model, num_heads=num_heads)
        self.global_pool = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Tanh(),
        )
        self.z_to_nodes = nn.Linear(d_model, d_model, bias=False)
        self.value_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, value_max + 1),
        )

    def masked_mean(self, h: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        weight = mask.unsqueeze(-1).float()
        denom = weight.sum(dim=1, keepdim=True).clamp_min(1.0)
        return (h * weight).sum(dim=1, keepdim=True) / denom

    def build_summary_tokens(self, h: torch.Tensor, kind_ids: torch.Tensor, node_mask: torch.Tensor) -> torch.Tensor:
        all_nodes = self.masked_mean(h, node_mask)
        variables = self.masked_mean(h, (kind_ids == NODE_KIND_TO_ID["variable"]) & (node_mask > 0))
        constants = self.masked_mean(h, (kind_ids == NODE_KIND_TO_ID["constant"]) & (node_mask > 0))
        operators = self.masked_mean(h, (kind_ids == NODE_KIND_TO_ID["operator"]) & (node_mask > 0))
        equations = self.masked_mean(h, (kind_ids == NODE_KIND_TO_ID["equation"]) & (node_mask > 0))
        return torch.cat([all_nodes, variables, constants, operators, equations], dim=1)

    def broadcast_summary(self, tokens: torch.Tensor, kind_ids: torch.Tensor, node_mask: torch.Tensor) -> torch.Tensor:
        global_tok = tokens[:, 0:1, :]
        variable_tok = tokens[:, 1:2, :]
        constant_tok = tokens[:, 2:3, :]
        operator_tok = tokens[:, 3:4, :]
        equation_tok = tokens[:, 4:5, :]

        context = global_tok.expand(-1, kind_ids.shape[1], -1)
        context = context + (kind_ids == NODE_KIND_TO_ID["variable"]).unsqueeze(-1) * variable_tok
        context = context + (kind_ids == NODE_KIND_TO_ID["constant"]).unsqueeze(-1) * constant_tok
        context = context + (kind_ids == NODE_KIND_TO_ID["operator"]).unsqueeze(-1) * operator_tok
        context = context + (kind_ids == NODE_KIND_TO_ID["equation"]).unsqueeze(-1) * equation_tok
        return context * node_mask.unsqueeze(-1)

    def gather_variable_states(self, h: torch.Tensor, variable_positions: torch.Tensor) -> torch.Tensor:
        safe_positions = variable_positions.clamp_min(0)
        gather_index = safe_positions.unsqueeze(-1).expand(-1, -1, h.shape[-1])
        gathered = torch.gather(h, dim=1, index=gather_index)
        return gathered

    def forward(self, batch: Dict[str, torch.Tensor], steps: int = 6) -> Dict[str, torch.Tensor]:
        kind_ids = batch["kind_ids"]
        op_ids = batch["op_ids"]
        eq_type_ids = batch["eq_type_ids"]
        numeric = batch["numeric"]
        node_mask = batch["node_mask"]
        adj = batch["adj"]
        rev_adj = batch["rev_adj"]
        left_adj = batch["left_adj"]
        right_adj = batch["right_adj"]
        variable_positions = batch["variable_positions"]

        h = (
            self.kind_emb(kind_ids)
            + self.op_emb(op_ids)
            + self.eq_type_emb(eq_type_ids)
            + self.numeric_proj(numeric)
        )
        h = h * node_mask.unsqueeze(-1)

        z = torch.zeros(h.shape[0], self.d_model, device=h.device)
        queen_context = torch.zeros_like(h)

        for step in range(steps):
            h = h + self.z_to_nodes(z).unsqueeze(1) + queen_context
            for worker in self.workers:
                h = worker(h, adj, rev_adj, left_adj, right_adj, node_mask)

            if step == 0 or (step % self.queen_period == 0):
                tokens = self.build_summary_tokens(h, kind_ids, node_mask)
                tokens = self.queen(tokens)
                queen_context = self.broadcast_summary(tokens, kind_ids, node_mask)

            z = self.global_pool(self.masked_mean(h, node_mask).squeeze(1))

        variable_states = self.gather_variable_states(h, variable_positions)
        value_logits = self.value_head(variable_states)
        return {
            "value_logits": value_logits,
            "node_states": h,
        }


model = AHRMathPrototype().to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")


## Training Loop

The loss is simple: for each masked variable node, predict the hidden integer value. This is a deliberately narrow objective, but it is enough to test whether the graph-native AHR learns to use equation structure.

In [ ]:
def move_batch(batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {
        key: value.to(DEVICE) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def compute_loss(output: Dict[str, torch.Tensor], batch: Dict[str, torch.Tensor]) -> torch.Tensor:
    logits = output["value_logits"]
    targets = batch["targets"]
    target_mask = batch["target_mask"]

    per_var_loss = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        reduction="none",
    ).reshape_as(target_mask)

    return (per_var_loss * target_mask).sum() / target_mask.sum().clamp_min(1.0)


@torch.no_grad()
def evaluate(loader: DataLoader, steps: int = 6) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_batches = 0
    total_var_correct = 0.0
    total_var_count = 0.0
    total_puzzle_correct = 0.0
    total_puzzle_count = 0.0

    for raw_batch in loader:
        batch = move_batch(raw_batch)
        output = model(batch, steps=steps)
        loss = compute_loss(output, batch)

        preds = output["value_logits"].argmax(dim=-1)
        target_mask_bool = batch["target_mask"] > 0
        per_var_correct = ((preds == batch["targets"]) & target_mask_bool).float().sum().item()
        per_var_count = target_mask_bool.float().sum().item()

        puzzle_correct = (((preds == batch["targets"]) | (~target_mask_bool)).all(dim=1)).float().sum().item()
        puzzle_count = batch["targets"].shape[0]

        total_loss += float(loss.item())
        total_batches += 1
        total_var_correct += per_var_correct
        total_var_count += per_var_count
        total_puzzle_correct += puzzle_correct
        total_puzzle_count += puzzle_count

    return {
        "loss": total_loss / max(total_batches, 1),
        "var_acc": total_var_correct / max(total_var_count, 1.0),
        "puzzle_acc": total_puzzle_correct / max(total_puzzle_count, 1.0),
    }


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

history = []
num_epochs = 8
train_steps = 5
valid_steps = 7

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    running_batches = 0

    for raw_batch in train_loader:
        batch = move_batch(raw_batch)
        optimizer.zero_grad()
        output = model(batch, steps=train_steps)
        loss = compute_loss(output, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += float(loss.item())
        running_batches += 1

    train_metrics = evaluate(train_loader, steps=valid_steps)
    valid_metrics = evaluate(valid_loader, steps=valid_steps)
    row = {
        "epoch": epoch,
        "train_loss": running_loss / max(running_batches, 1),
        "train_var_acc": train_metrics["var_acc"],
        "train_puzzle_acc": train_metrics["puzzle_acc"],
        "valid_loss": valid_metrics["loss"],
        "valid_var_acc": valid_metrics["var_acc"],
        "valid_puzzle_acc": valid_metrics["puzzle_acc"],
    }
    history.append(row)
    print(row)


In [ ]:
history


## Qualitative Inspection

The final check is qualitative: inspect predicted variable assignments against the exact hidden solution and look at the equations that generated the puzzle.

In [ ]:
@torch.no_grad()
def inspect_predictions(raw_puzzles: List[Dict], loader: DataLoader, num_examples: int = 5):
    model.eval()
    batch = move_batch(next(iter(loader)))
    output = model(batch, steps=8)
    preds = output["value_logits"].argmax(dim=-1).cpu()
    targets = batch["targets"].cpu()
    target_mask = batch["target_mask"].cpu() > 0

    for i in range(min(num_examples, preds.shape[0])):
        truth = {
            f"x{j}": int(targets[i, j])
            for j in range(MAX_VARS)
            if target_mask[i, j]
        }
        guess = {
            f"x{j}": int(preds[i, j])
            for j in range(MAX_VARS)
            if target_mask[i, j]
        }
        print("=" * 80)
        print("truth:", truth)
        print("guess:", guess)
        print("equations:")
        for equation in raw_puzzles[i]["equations"]:
            print("  ", equation["equation_pretty"])


inspect_predictions(valid_raw, valid_loader, num_examples=5)


## Where To Go Next

If this prototype learns anything useful, the next upgrades are clear:

1. Replace the synthetic-only front end with a LaTeX parser that emits graph candidates.
2. Add problem-family routing and specialized expert heads.
3. Let the queen choose actions such as simplification, branching, or symbolic verification.
4. Add a candidate-answer beam and rerank with tool-backed consistency checks.
5. Train on intermediate traces, not just final hidden-variable recovery.

That is the path from a graph-native AHR toy model to an actual AIMO solver stack.